# Klasifikacija vesti (fake / real)

## Biblioteke

In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



## Preuzimanje dataseta

In [20]:
!kaggle datasets download -d clmentbisaillon/fake-and-real-news-dataset -p data --unzip

Dataset URL: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
License(s): CC-BY-NC-SA-4.0




  0%|          | 0.00/41.0M [00:00<?, ?B/s]
  2%|▏         | 1.00M/41.0M [00:00<00:31, 1.34MB/s]
  5%|▍         | 2.00M/41.0M [00:01<00:20, 2.04MB/s]
  7%|▋         | 3.00M/41.0M [00:01<00:16, 2.44MB/s]
 10%|▉         | 4.00M/41.0M [00:01<00:14, 2.65MB/s]
 12%|█▏        | 5.00M/41.0M [00:02<00:13, 2.82MB/s]
 15%|█▍        | 6.00M/41.0M [00:02<00:12, 3.00MB/s]
 17%|█▋        | 7.00M/41.0M [00:02<00:11, 3.14MB/s]
 20%|█▉        | 8.00M/41.0M [00:03<00:10, 3.22MB/s]
 22%|██▏       | 9.00M/41.0M [00:03<00:10, 3.28MB/s]
 24%|██▍       | 10.0M/41.0M [00:03<00:09, 3.34MB/s]
 27%|██▋       | 11.0M/41.0M [00:03<00:09, 3.36MB/s]
 29%|██▉       | 12.0M/41.0M [00:04<00:09, 3.29MB/s]
 32%|███▏      | 13.0M/41.0M [00:04<00:08, 3.28MB/s]
 34%|███▍      | 14.0M/41.0M [00:04<00:08, 3.17MB/s]
 37%|███▋      | 15.0M/41.0M [00:05<00:08, 3.21MB/s]
 39%|███▉      | 16.0M/41.0M [00:05<00:08, 3.21MB/s]
 41%|████▏     | 17.0M/41.0M [00:05<00:08, 3.03MB/s]
 44%|████▍     | 18.0M/41.0M [00:06<00:07, 3.17MB/s]
 

## Ucitavanje podataka

In [21]:
import pandas as pd

fake = pd.read_csv("data/Fake.csv"); fake["label"] = 1
true = pd.read_csv("data/True.csv"); true["label"] = 0
df = pd.concat([fake, true], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
df.shape

(44898, 5)

## Pregled podataka

In [24]:
print(df.info())
print(df.label.value_counts())

df.duplicated(subset="text").sum()

df.text.str.strip().eq("").sum()

print(df.subject.value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   title    44898 non-null  str  
 1   text     44898 non-null  str  
 2   subject  44898 non-null  str  
 3   date     44898 non-null  str  
 4   label    44898 non-null  int64
dtypes: int64(1), str(4)
memory usage: 112.3 MB
None
label
1    23481
0    21417
Name: count, dtype: int64
subject
politicsNews       11272
worldnews          10145
News                9050
politics            6841
left-news           4459
Government News     1570
US_News              783
Middle-east          778
Name: count, dtype: int64


## Tokenizacija i podela na skupove

In [27]:
!pip install scikit-learn

In [33]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(df.subject, df.label, stratify = df.label, random_state=42)

tok = Tokenizer(num_words = 3000, oov_token = "<OOV>")


tok.fit_on_texts(X_train)

Xtre = pad_sequences(tok.texts_to_sequences(X_train), 300, truncating="post")
Xpre = pad_sequences(tok.texts_to_sequences(X_test), 300, truncating="post")


## CNN model

In [34]:

from tensorflow.keras import layers, models

cnn = models.Sequential([
    layers.Embedding(3000, 128, input_length=300),
    layers.Conv1D(128, 5, activation="relu"),
    layers.GlobalMaxPooling1D(),
    layers.Dropout(0.5),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])
cnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

c:\Users\bogda\miniconda3\envs\ml\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


## LSTM model

In [35]:
lstm = models.Sequential([
    layers.Embedding(3000, 128, input_length=300),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])
lstm.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

## Treniranje

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
hist = cnn.fit(Xtre, y_train, validation_split=0.1, epochs=10, batch_size=64, callbacks=[es])

## Dotreniranje transformera

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256, padding="max_length")

args = TrainingArguments(
    output_dir="out", num_train_epochs=2, learning_rate=2e-5,
    per_device_train_batch_size=32, eval_strategy="epoch", fp16=True)